# Data Transformations

## Overview
Transformations are applied before analysis to satisfy parametric assumptions — principally normality of residuals and homogeneity of variance. The choice of transformation should be driven by the data-generating process or diagnostic plots, never by the desire to achieve significance.

| Transformation | Formula | Best for | Back-transform |
|---|---|---|---|
| Square root | `sqrt(y)` | Counts, Poisson-like | `estimate^2` |
| Log | `log(y)` | Rates, ratios, concentrations | `exp(estimate)` |
| Log + 1 | `log(y + 1)` | Counts with zeros | `exp(estimate) - 1` |
| Arcsine-sqrt | `asin(sqrt(p))` | Proportions [0,1] | `sin(estimate)^2` |
| Box-Cox | `(y^λ - 1)/λ` | Data-driven optimal | `(estimate·λ + 1)^(1/λ)` |

**Underwood (1997):** transform to meet assumptions, not to get a significant result.

---

In [ ]:
library(tidyverse); library(MASS); library(car)
set.seed(42)
n <- 80

# Count data — seabird abundance
counts <- data.frame(
  species   = rep(c("Tern","Gannet"), each = n/2),
  abundance = c(rpois(n/2, 8), rpois(n/2, 15))
)
# Concentration data — right-skewed nitrogen (log-normal)
nutrients <- data.frame(
  depth    = rep(c("Surface","Bottom"), each = 40),
  nitrogen = c(rlnorm(40, 1.5, 0.8), rlnorm(40, 2.0, 0.9))
)
# Proportion data — vegetation cover
props <- data.frame(
  treatment = rep(c("Control","Fertilised","Grazed"), each = 25),
  cover     = c(rbeta(25,2,5), rbeta(25,5,3), rbeta(25,3,4))
)
cat("Datasets: counts, nutrients, props\n")

---
## Square-root for counts; log for concentrations

In [ ]:
par(mfrow = c(2, 4))
# Raw counts
hist(counts$abundance, main="Raw counts", col="steelblue", xlab="Abundance")
qqnorm(counts$abundance, main="QQ: raw"); qqline(counts$abundance)
# sqrt transformed
counts$sqrt_abund <- sqrt(counts$abundance)
hist(counts$sqrt_abund, main="sqrt(counts)", col="coral", xlab="sqrt(Abundance)")
qqnorm(counts$sqrt_abund, main="QQ: sqrt"); qqline(counts$sqrt_abund)
# Log nutrients
nutrients$log_n <- log(nutrients$nitrogen)
hist(nutrients$nitrogen, main="Raw N", col="steelblue", xlab="N mg/L")
qqnorm(nutrients$nitrogen, main="QQ: raw N"); qqline(nutrients$nitrogen)
hist(nutrients$log_n, main="log(N)", col="coral", xlab="log N")
qqnorm(nutrients$log_n, main="QQ: log N"); qqline(nutrients$log_n)
par(mfrow=c(1,1))

# Back-transformation: geometric mean on original scale
m_log <- lm(log_n ~ depth, data=nutrients)
b <- coef(m_log)
cat("Geometric mean N — Bottom:", round(exp(b[1]),2), "mg/L\n")
cat("Surface:Bottom ratio:      ", round(exp(b[2]),3), "\n")
cat("(Back-transform the estimate, NOT the mean of back-transformed values)\n")

---
## Arcsine-sqrt for proportions; Box-Cox for optimal lambda

In [ ]:
# Arcsine-sqrt
props$asin_cover <- asin(sqrt(props$cover))
par(mfrow=c(1,3))
hist(props$cover,      main="Raw proportions",   col="steelblue")
hist(props$asin_cover, main="asin(sqrt(cover))", col="coral")

# Box-Cox for data-driven lambda selection
bc <- MASS::boxcox(lm(nitrogen ~ depth, data=nutrients),
                   plotit=TRUE, lambda=seq(-2,2,0.1))
lambda_opt <- bc$x[which.max(bc$y)]
par(mfrow=c(1,1))
cat("Optimal Box-Cox lambda:", round(lambda_opt,3),"\n")
cat("  lambda ~  0  → log transformation\n")
cat("  lambda ~  0.5 → square root\n")
cat("  lambda ~  1  → no transformation needed\n")
cat("\nModern alternative for proportions: beta regression (betareg)\n")
cat("Modern alternative for counts: GLM with Poisson/NB family\n")
cat("Transformations are less necessary when using appropriate GLMs.\n")

---
## Common Pitfalls

**1. Transforming to achieve significance, not to meet assumptions**
Always diagnose assumptions on raw data first. Trying multiple transformations until p < 0.05 is p-hacking — the transformation must be justified by the data-generating process or diagnostic plots.

**2. Forgetting to back-transform estimates and CIs**
A mean of 2.3 on the log scale is meaningless to a reader. Back-transform the model estimate and its CI: `exp(estimate)` gives the geometric mean. The arithmetic mean of back-transformed values is not the back-transformed mean.

**3. Using arcsine-sqrt when zeros or ones are present**
`asin(sqrt(0))` = 0 and `asin(sqrt(1))` = π/2 are valid, but many texts warn about boundary values distorting the transformation. If exact 0s and 1s are common, use a zero-one inflated beta model instead.

**4. Applying log to data with zeros**
`log(0)` = -Inf. Adding a constant (`log(y+1)`) is arbitrary. For count data with genuine zeros, use a Poisson or negative binomial GLM directly.

**5. Assuming the same transformation works for all groups**
A single transformation may fix skewness in one group while distorting another. Always check diagnostics within each group separately.

**6. Reporting results only on the transformed scale**
For biological interpretation, report estimated means and CIs on the original scale. Always state the transformation used and how estimates were back-transformed.


---
*r_methods_library - Samantha McGarrigle*